<a href="https://colab.research.google.com/github/jaqueuchoab/dados-divas/blob/main/compilar_arquivos_enade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Algoritmo para junção dos arquivos da base do ENADE**

Uma vez que a base do ENADE é fragmentada em muitos arquivos, este
algoritmo foi desenvolvido com auxílio da *Inteligência Artificial Claude
Sonnet 4.6* para realizar a junção dos arquivos de forma que seja possível
relacionar os dados a nível de curso e instituição.

É de suma importância destacar que, apesar de cada linha do ENADE
corresponder a um aluno, **não é possível identificar os estudantes na base**.
Nesse contexto, as linhas não definem o perfil individual de um aluno e a
base não deve ser utilizada para esse fim, pois as colunas de cada arquivo
foram unidas horizontalmente com base no `CO_CURSO`, sem a perspectiva de
combinação de informações entre arquivos distintos.

**Importações de bibliotecas necessárias:**
- `pandas` — manipulação e compilação dos dados
- `glob` — busca automática dos arquivos na pasta
- `os` — criação automática da pasta de saída
- `re` — extração do número do nome de cada arquivo para ordenação numérica

In [8]:
import pandas as pd
import glob
import os
import re

**Configurações**

Essa seção permite limitar os arquivos e colunas que irão ser unidos no arquivos final.

Caso nenhum deva ser excluido, basta deixar as constantes vazias e a limitação para teste como 0.



In [9]:
# Arquivos a ignorar
ARQUIVOS_IGNORAR = [
]

# Arquivos a incluir
ARQUIVOS_INCLUIR = [
    'microdados2022_arq1.txt',
    'microdados2022_arq3.txt',
]

# Colunas para incluir
COLUNAS_INCLUIR = [
    'CO_CURSO',
    'CO_MUNIC_CURSO',
    'CO_IES',
    'TP_PR_GER',
    'TP_PR_OB_FG',
    'TP_PR_DI_FG',
    'TP_PR_OB_CE',
    'TP_PR_DI_CE',
    'NT_GER',
    'NT_FG',
    'NT_OBJ_FG',
    'NT_DIS_FG',
    'NT_CE',
    'NT_OBJ_CE',
    'NT_DIS_CE',
    'CO_RS_I1',
    'CO_RS_I2'
]

# Colunas a ignorar
COLUNAS_IGNORAR = [
    # 'NU_ANO'
]

# IDEAL P/TESTE: limitar quantidade de arquivos (0 = todos)
LIMITE_ARQUIVOS = 0

**Passo 1: Carregamento do arquivo 1**

O arquivo 1 é o único que contém o código da instituição (`CO_IES`) junto
ao código do curso (`CO_CURSO`). Ele serve como base inicial da compilação
e como chave de identificação institucional para todos os demais arquivos.

O tratamento do BOM (`ï»¿`) remove um caractere invisível que aparece no
nome da primeira coluna quando o CSV é salvo com encoding `utf-8-sig`.

In [10]:
# Carregamento do arquivo 1 do ano para junção
arq1 = pd.read_csv('/content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2022/microdados2022_arq1.txt', sep=';', encoding='latin-1')
# Tratamento do BOM (Byte Order Mark)
arq1.columns = arq1.columns.str.replace('ï»¿', '', regex=False).str.strip()
# Ordenação da coluna curso do menor para o maior
arq1 = arq1.sort_values('CO_CURSO').reset_index(drop=True)

# Remoção das colunas definidas em COLUNAS_IGNORAR, se existirem no arquivo
colunas_remover_arq1 = [c for c in COLUNAS_IGNORAR if c in arq1.columns]
if colunas_remover_arq1:
    arq1 = arq1.drop(columns=colunas_remover_arq1)
    print(f"Colunas removidas do arquivo 1: {colunas_remover_arq1}")

print(f"Arquivo 1: {len(arq1)} linhas, {len(arq1.columns)} colunas")

# Chave para validação
chave = arq1[['CO_IES', 'CO_CURSO', 'CO_MUNIC_CURSO']].drop_duplicates()
print(f"\nCursos únicos:       {chave['CO_CURSO'].nunique()}")
print(f"Instituições únicas: {chave['CO_IES'].nunique()}")
print(f"Municípios únicos:  {chave['CO_MUNIC_CURSO'].nunique()}")
print(f"Nulos em CO_IES:     {chave['CO_IES'].isna().sum()}")

Arquivo 1: 594013 linhas, 10 colunas

Cursos únicos:       9896
Instituições únicas: 1749
Municípios únicos:  782
Nulos em CO_IES:     0


**Passo 2: Listagem e ordenação dos arquivos**

Os arquivos restantes são listados e ordenados **numericamente** — não
alfabeticamente — para garantir o carregamento sequencial.

O arquivo 1 é excluído da lista pois já foi carregado separadamente.
Se `LIMITE_ARQUIVOS` for maior que zero, apenas os primeiros N arquivos
serão processados (útil para testes).

In [11]:
# Procura dos arquivos disponíveis para junção - importante que esteja apenas os necessários
arquivos = glob.glob('/content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2022/microdados2022_arq*.txt')
# Exclusão do arq1 - visto que ele é a base não é necessário recarregar ele
arquivos = [a for a in arquivos if 'microdados2022_arq1.txt' not in a]

# Ordenar pelo número extraído do nome do arquivo - garantindo consistência na sequência de junção das colunas
arquivos = sorted(arquivos, key=lambda x: int(re.search(r'(\d+)', os.path.basename(x)).group()))

# Aplicar filtro de inclusão de arquivos (tem prioridade se preenchido)
if ARQUIVOS_INCLUIR:
    arquivos = [a for a in arquivos if os.path.basename(a) in ARQUIVOS_INCLUIR]
# Senão aplicar filtro de exclusão
elif ARQUIVOS_IGNORAR:
    arquivos = [a for a in arquivos if os.path.basename(a) not in ARQUIVOS_IGNORAR]

# Aplicar limite de teste
if LIMITE_ARQUIVOS > 0:
    arquivos = arquivos[:LIMITE_ARQUIVOS]

print(f"\nArquivos encontrados ({len(arquivos)}):")
for a in arquivos:
    print(f"  {a}")


Arquivos encontrados (1):
  /content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2022/microdados2022_arq3.txt


**Passo 3: Base inicial**

O arquivo 1 se torna a base inicial da compilação. Todos os demais arquivos
serão adicionados a ele horizontalmente, coluna a coluna.

In [12]:
# Cópia do arquivo 1 - base da junção
base = arq1.copy()

# Aplicar filtro de inclusão e exclusão de colunas no arquivo 1
if COLUNAS_INCLUIR:
    colunas_manter = ['CO_CURSO', 'CO_IES'] + [c for c in COLUNAS_INCLUIR if c in base.columns]
    # Remoção das colunas duplicatas uma vez que há uma junção manual de CO_CURSO e CO_IES
    colunas_manter = list(dict.fromkeys(colunas_manter))
    base = base[colunas_manter]
elif COLUNAS_IGNORAR:
    colunas_remover = [c for c in COLUNAS_IGNORAR if c in base.columns]
    if colunas_remover:
        base = base.drop(columns=colunas_remover)

print(f"\nBase inicial (arquivo 1): {len(base)} linhas, {len(base.columns)} colunas")


Base inicial (arquivo 1): 594013 linhas, 3 colunas


**Passo 4: Compilação dos arquivos**

Para cada arquivo da lista:
1. Carrega o arquivo e corrige o encoding das colunas
2. Ordena pelo `CO_CURSO` para garantir alinhamento com a base
3. Remove colunas definidas em `COLUNAS_IGNORAR`
4. Identifica colunas que ainda não existem na base
5. Adiciona apenas as colunas novas horizontalmente com `concat(axis=1)`

Colunas já presentes na base são ignoradas automaticamente para evitar
duplicatas — por exemplo, `NU_ANO` que aparece em vários arquivos.

In [13]:
for arq in arquivos:
    df = pd.read_csv(arq, sep=';', encoding='latin-1')
    df.columns = df.columns.str.replace('ï»¿', '', regex=False).str.strip()
    df = df.sort_values('CO_CURSO').reset_index(drop=True)

    # Aplicar filtro de colunas
    if COLUNAS_INCLUIR:
        colunas_manter = ['CO_CURSO'] + [c for c in COLUNAS_INCLUIR if c in df.columns]
        colunas_manter = list(dict.fromkeys(colunas_manter))
        df = df[colunas_manter]
    elif COLUNAS_IGNORAR:
        colunas_remover = [c for c in COLUNAS_IGNORAR if c in df.columns]
        if colunas_remover:
            df = df.drop(columns=colunas_remover)

    # Comparar colunas de df e base e ignorar colunas que já existem na base para evitar duplicidade
    colunas_novas = [c for c in df.columns if c not in base.columns]

    if not colunas_novas:
        print(f"{arq} → sem colunas novas, ignorado")
        continue

    # Junção da base (arquivo 1) com o df (do arquivo da vez, com colunas novas) de forma horizontal (axis=1)
    base = pd.concat([base, df[colunas_novas]], axis=1)

    print(f"{arq} adicionado → {len(base)} linhas, {len(base.columns)} colunas")

/tmp/ipykernel_715/3168117839.py:2: DtypeWarning: Columns (12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=';', encoding='latin-1')


/content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2022/microdados2022_arq3.txt adicionado → 594013 linhas, 17 colunas


**Passo 5: Validação do resultado**

Verificações básicas para confirmar que a compilação foi bem-sucedida:

- O número de linhas deve permanecer igual ao número de linhas do arquivo 1, mostrado no ínicio
- O número de cursos únicos deve bater com o arquivo 1
- O número de instituições únicas deve bater com o arquivo 1
- Nulos em `CO_IES` devem ser zero

In [14]:
print(f"\n=== RESULTADO FINAL ===")
print(f"Linhas:              {len(base)}")
print(f"Colunas:             {len(base.columns)}")
print(f"Cursos únicos:       {base['CO_CURSO'].nunique()}")
print(f"Instituições únicas: {base['CO_IES'].nunique()}")
print(f"Nulos em CO_IES:     {base['CO_IES'].isna().sum()}")


=== RESULTADO FINAL ===
Linhas:              594013
Colunas:             17
Cursos únicos:       9896
Instituições únicas: 1749
Nulos em CO_IES:     0


**Passo 6: Exportação**

Uma pasta `data` é criada e dentro dela está o arquivo compilado.

A base compilada é salva em dois formatos:

- **Parquet** — formato eficiente para uso no pipeline de integração de bases
- **CSV** — formato aberto para inspeção visual e interoperabilidade

O encoding `utf-8-sig` no CSV garante que acentos apareçam corretamente ao abrir no Excel.


In [15]:
# Criação da pasta de output
os.makedirs('data', exist_ok=True)

# Aqui é possivel redefinir o nome do arquivo de saída conforme a necessidade
base.to_parquet('data/enade_compilado.parquet', index=False)
base.to_csv('data/enade_compilado.csv', sep=';',
            encoding='utf-8-sig', index=False)

print(f"\nBase salva em data/")


Base salva em data/
